# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

In [1]:
#Professor I have changed my research topic to something else
#New Research question: What predicts a country's population density?
#Sources: https://www.youtube.com/watch?v=wQqLVioM0Pw

import requests
import pandas as pd


In [2]:
#I am making my API link into a variable because I do not want to type the entire link out
url = 'https://restcountries.com/v3.1/all?fields=name,population,area,region,subregion,languages,borders'

print("Calling the REST Countries API...")
response = requests.get(url)

print(f"Status code: {response.status_code}")

Calling the REST Countries API...
Status code: 200


In [3]:
# I am parsing the response into a Python list in this cell
countries_raw = response.json()

print(f"Total countries returned: {len(countries_raw)}") #Source: https://realpython.com/len-python-function/
print("Available fields:", list(countries_raw[0].keys()))

Total countries returned: 250
Available fields: ['name', 'languages', 'region', 'subregion', 'borders', 'area', 'population']


In [4]:
# Here I made an emptpy list called record and I want to basically loop thorugh all the countries and get the variables I need.
records = []

for country in countries_raw:
#The Variables I want to use are name of the country, population, area, region, and subregion.
    name = country.get('name', {}).get('common', None)
    population = country.get('population', None)
    area = country.get('area', None)
    region = country.get('region', None)
    subregion = country.get('subregion', None)

    languages = country.get('languages', {})
    num_languages = len(languages) if languages else 0

    borders = country.get('borders', [])
    num_borders = len(borders) if borders else 0

    records.append({
        'country':       name,
        'population':    population,
        'area_km2':      area,
        'region':        region,
        'subregion':     subregion,
        'num_languages': num_languages,
        'num_borders':   num_borders
    })

# This lets me turn all of this into a dataframe
df = pd.DataFrame(records)

print(f"DataFrame shape: {df.shape}")
df.head(10)

DataFrame shape: (250, 7)


,country,population,area_km2,region,subregion,num_languages,num_borders
0,Cook Islands,15040,236.0,Oceania,Polynesia,2,0
1,Guinea,14363931,245857.0,Africa,Western Africa,1,6
2,Christmas Island,1692,135.0,Oceania,Australia and New Zealand,1,0
3,Togo,8095498,56785.0,Africa,Western Africa,1,3
4,Taiwan,23317031,36197.0,Asia,Eastern Asia,1,0
5,Kyrgyzstan,7281800,199951.0,Asia,Central Asia,2,4
6,Suriname,616500,163820.0,Americas,South America,1,3
7,Dominican Republic,10771504,48671.0,Americas,Caribbean,1,1
8,Guatemala,18079810,108889.0,Americas,Central America,1,4
9,Algeria,47400000,2381741.0,Africa,Northern Africa,1,7


In [5]:
#I just want to see if I have any missing data because if some cell have a zero in them then that could ruining the results.
print("Data types:")
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum())

Data types:
country           object
population         int64
area_km2         float64
region            object
subregion         object
num_languages      int64
num_borders        int64
dtype: object

Missing values:
country          0
population       0
area_km2         0
region           0
subregion        0
num_languages    0
num_borders      0
dtype: int64


In [6]:
#I found all the cells with missing data in the coding cell before, so here I just want to remove them. 
# I wanted to drop rows with missing population or area because I dont want the results to be affected.
rows_before = len(df)
df = df.dropna(subset=['population', 'area_km2'])
print(f"Rows dropped (missing values): {rows_before - len(df)}")

# remove if area = 0 
df = df[df['area_km2'] > 0]

# remove if population = 0 
df = df[df['population'] > 0]

# reseting the index
df = df.reset_index(drop=True)

print(f"Rows remaining after cleaning: {len(df)}")
df.head()

Rows dropped (missing values): 0
Rows remaining after cleaning: 245


,country,population,area_km2,region,subregion,num_languages,num_borders
0,Cook Islands,15040,236.0,Oceania,Polynesia,2,0
1,Guinea,14363931,245857.0,Africa,Western Africa,1,6
2,Christmas Island,1692,135.0,Oceania,Australia and New Zealand,1,0
3,Togo,8095498,56785.0,Africa,Western Africa,1,3
4,Taiwan,23317031,36197.0,Asia,Eastern Asia,1,0


In [7]:
# This is the part where I am creating my new variable
df['population_density'] = df['population'] / df['area_km2']
df['population_density'] = df['population_density'].round(2)

# Top 10 most densely populated countries
df[['country', 'population', 'area_km2', 'population_density']].sort_values(
    'population_density', ascending=False
).head(10)

,country,population,area_km2,population_density
151,Macau,685900,30.00,22863.33
135,Monaco,38423,2.02,19021.29
26,Singapore,6110200,710.00,8605.92
174,Hong Kong,7527500,1104.00,6818.39
129,Gibraltar,38000,6.00,6333.33
153,Bahrain,1594654,765.00,2084.52
171,Malta,574250,316.00,1817.25
125,Vatican City,882,0.49,1800.00
187,Maldives,515132,300.00,1717.11
144,Sint Maarten,41349,34.00,1216.15


In [8]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print()
print("Column names:", list(df.columns))
print()
df.describe().round(2)

Rows: 245
Columns: 8

Column names: ['country', 'population', 'area_km2', 'region', 'subregion', 'num_languages', 'num_borders', 'population_density']



,population,area_km2,num_languages,num_borders,population_density
count,2.450000e+02,245.00,245.00,245.00,245.00
mean,3.273263e+07,613286.75,1.67,2.64,445.85
std,1.332300e+08,1930122.03,1.40,2.70,2051.28
min,3.500000e+01,0.49,0.00,0.00,0.00
25%,2.923540e+05,1628.00,1.00,0.00,31.12
50%,5.343630e+06,69700.00,1.00,2.00,89.76
75%,2.073426e+07,390757.00,2.00,4.00,242.27
max,1.417492e+09,17098246.00,15.00,16.00,22863.33


In [10]:
df.to_csv('final_country_dataset.csv', index=False)
print(f"Total countries: {len(df)}")

Dataset saved to 'final_country_dataset.csv'
Total countries: 245
